# Lab 07: Diffusion Visualization

**Module 07** | Read `notes/07-diffusion-theory.pdf` before running this notebook.

Visualize the forward noising process and a linear beta schedule on MNIST digits, the foundation for DDPM training.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Runtime device

PyTorch can run on your NVIDIA GPU through CUDA or fall back to CPU. GPU execution moves matrix operations off the host and typically speeds up neural network training by a large factor. The next cell detects what is available and prints the active device so you know whether to expect fast training or should keep batch sizes small.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA enabled, {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available, using CPU. Labs still run; training may take longer.")


## Forward diffusion on MNIST

Denoising diffusion models gradually corrupt data by adding Gaussian noise over `T` timesteps. The **forward process** is Markovian: at each step a small amount of noise is blended in according to a variance schedule `beta_t`. After enough steps the image approaches pure noise.

Understanding the forward process is prerequisite to DDPM training in Lab 08: the network learns to reverse these corruption steps.


### Step 1: Build the beta schedule

We use a **linear beta schedule**: `beta_t` increases evenly from a small start value to a larger end value. From `beta_t` we derive `alpha_t = 1 - beta_t` and the cumulative product `alpha_bar_t = prod(alpha_i)`, which lets us jump directly from clean image `x_0` to any noisy `x_t`:

`x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * epsilon` with `epsilon ~ N(0, I)`.

The plots and printed statistics below summarize how quickly signal is destroyed across timesteps.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torchvision import datasets, transforms

ROOT = Path("..").resolve()
DATA_DIR = ROOT / "datasets" / "mnist"
T = 500

betas = torch.linspace(1e-4, 0.02, T)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

print("Diffusion schedule statistics")
print(f"  T (timesteps):     {T}")
print(f"  beta_1:            {betas[0]:.6f}")
print(f"  beta_T:            {betas[-1]:.4f}")
print(f"  alpha_bar_1:       {alpha_bars[0]:.4f}")
print(f"  alpha_bar_T:       {alpha_bars[-1]:.6f}")
print(f"  Signal at t=100:   {alpha_bars[99]:.4f} (fraction of x_0 variance retained)")
print(f"  Signal at t=250:   {alpha_bars[249]:.4f}")
print(f"  Signal at t=500:   {alpha_bars[499]:.6f} (nearly pure noise)")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(betas.numpy())
axes[0].set_title("Linear beta schedule")
axes[0].set_xlabel("Timestep t")
axes[0].set_ylabel("beta_t")
axes[0].grid(True, alpha=0.3)

axes[1].plot(alpha_bars.numpy())
axes[1].set_title("Cumulative alpha_bar = prod(1 - beta)")
axes[1].set_xlabel("Timestep t")
axes[1].set_ylabel("alpha_bar_t")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Step 2: Load one MNIST digit

We pick the first training image and record its label. All forward-noise steps reuse the same underlying Gaussian noise `epsilon` so changes across timesteps reflect the schedule, not resampling.


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

train_set = datasets.MNIST(root=str(DATA_DIR), train=True, download=True, transform=transform)
x0, label = train_set[0]
x0 = x0.unsqueeze(0).to(device)

print("Selected MNIST example")
print(f"  Dataset index: 0")
print(f"  Digit label:   {label}")
print(f"  Shape:         {tuple(x0.shape)}")
print(f"  Pixel mean:    {x0.mean():.4f}")
print(f"  Pixel std:     {x0.std():.4f}")


### Step 3: Apply forward diffusion at selected timesteps

We corrupt the digit at timesteps `[0, 50, 100, 200, 500]`. Early steps look almost unchanged; by step 500 structure dissolves into noise. Pixel mean and std drift toward those of pure Gaussian noise (mean 0, std 1 in normalized space).

Read the printed table: as `alpha_bar_t` shrinks, the image loses contrast and fine detail.


In [ ]:
timesteps = [0, 50, 100, 200, 500]
noisy_images = []

torch.manual_seed(42)
noise = torch.randn_like(x0)

print(f"{'t':>4} | {'alpha_bar':>10} | {'mean':>8} | {'std':>8} | interpretation")
print("-" * 70)

for t in timesteps:
    if t == 0:
        xt = x0
        ab = torch.tensor(1.0)
        note = "clean image (no noise added)"
    else:
        ab = alpha_bars[t - 1]
        ab_dev = ab.to(device)
        xt = torch.sqrt(ab_dev) * x0 + torch.sqrt(1.0 - ab_dev) * noise
        if t <= 100:
            note = "mostly signal, slight blur"
        elif t <= 200:
            note = "mixed signal and noise"
        else:
            note = "dominated by noise"
    stats = xt.detach().cpu()
    print(
        f"{t:4d} | {ab.item():10.6f} | {stats.mean():8.4f} | {stats.std():8.4f} | {note}"
    )
    noisy_images.append(stats)

fig, axes = plt.subplots(1, len(timesteps), figsize=(12, 3))
for ax, t, img in zip(axes, timesteps, noisy_images):
    disp = img.squeeze().numpy() * 0.5 + 0.5
    ax.imshow(disp, cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"t = {t}")
    ax.axis("off")
plt.suptitle(f"Forward diffusion of MNIST digit {label}")
plt.tight_layout()
plt.show()


### Step 4: Interpret noise level

When `alpha_bar_t` is close to 1, `x_t` is almost identical to `x_0`. When `alpha_bar_t` approaches 0, the sqrt terms weight noise heavily and the digit becomes unrecognizable. DDPM training samples random `t` values across this entire range so the network learns denoising at every signal-to-noise ratio.
